# Shiurim Otzar HaTorah → tu Google Drive

Pone **todos los shiurim de Otzar HaTorah** (9,911, unos **184 GB**) en tu Google Drive, acomodados por tema:

`1. Parashá` · `2. Jaguim` · `3. Halajá` (por Shulján Aruj) · `4. Musar y Emuná` (por Rab) · `5. Tanaj` · `6. Shmirat HaLashon`

Todo pasa en los servidores de Google: **no usa tu internet ni tu compu**, solo hay que dejar esta pestaña abierta.

**Cómo:**
1. Tu cuenta de Google necesita **200 GB o más** (one.google.com).
2. En la segunda casilla, **pega el link de tu carpeta de Drive** (o déjalo vacío: busca sola la carpeta "Colelim Mexico").
3. Arriba: **Entorno de ejecución → Ejecutar todo** (o Ctrl+F9). Acepta los permisos de Google Drive.
4. Deja la pestaña abierta. Si se corta (Colab corta a las ~12 h o si la compu se duerme), vuelve a darle **Ejecutar todo**: sigue donde se quedó, no repite nada.

Quedan dentro de tu carpeta (la del link, o **Colelim Mexico**), con un `ÍNDICE.txt` arriba. Lo que ya tenías ahí no se toca.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#@title ⬇️ Pega aquí el link de tu carpeta de Drive (o déjalo vacío y la busca por nombre)
LINK_DE_TU_CARPETA = "" #@param {type:"string"}
CARPETA = "Colelim Mexico" #@param {type:"string"}
import gzip, json, os, shutil, time, urllib.request, concurrent.futures as cf, threading
PLAN_URL = "https://raw.githubusercontent.com/jaimeatach/jaburahalaja/claude/shiurim-halachot-shabbat-vriig5/tools/otzar/drive_sammy_plan.json.gz"
import re, unicodedata
def _plano(s):
    s = unicodedata.normalize("NFKD", s)
    return re.sub(r"[^a-z0-9]", "", "".join(c for c in s if not unicodedata.combining(c)).lower())
def _parecido(nombre):
    a, b = _plano(nombre), _plano(CARPETA)
    return a == b or (a.startswith("colelim") and re.sub(r"^colelim", "", a) in ("mexico", "meksiko", "mejico", "mexiko", "meksico"))
def hallar_carpeta():
    raices = ["/content/drive/MyDrive"] + [os.path.join("/content/drive/Shareddrives", d) for d in
             (os.listdir("/content/drive/Shareddrives") if os.path.isdir("/content/drive/Shareddrives") else [])]
    hallados, nivel = [], [(r, 0) for r in raices]
    while nivel:
        d, prof = nivel.pop(0)
        if _parecido(os.path.basename(d)):
            hallados.append(d); continue
        if prof < 2:
            try:
                nivel += [(os.path.join(d, h), prof + 1) for h in sorted(os.listdir(d)) if os.path.isdir(os.path.join(d, h))]
            except OSError:
                pass
    return hallados
def carpeta_del_link(link):
    """El link de Drive solo trae el id de la carpeta: con la API de Drive se sube de padre en padre
    hasta la raíz y se arma la ruta dentro de /content/drive."""
    m = re.search(r"/folders/([\w-]+)|[?&]id=([\w-]+)", link)
    if not m:
        print("Ese link no es de una carpeta de Drive (tiene que decir /folders/…)."); raise SystemExit
    fid = m.group(1) or m.group(2)
    from google.colab import auth
    from googleapiclient.discovery import build
    auth.authenticate_user()
    api = build("drive", "v3")
    raiz = api.files().get(fileId="root", fields="id").execute()["id"]
    partes, actual = [], fid
    while True:
        f = api.files().get(fileId=actual, fields="id,name,parents,driveId,mimeType,shortcutDetails",
                            supportsAllDrives=True).execute()
        if f.get("mimeType") == "application/vnd.google-apps.shortcut":
            actual = f["shortcutDetails"]["targetId"]; continue
        if f["id"] == raiz:
            return os.path.join("/content/drive/MyDrive", *reversed(partes))
        if f.get("driveId") and f["id"] == f["driveId"]:
            nombre = api.drives().get(driveId=f["driveId"]).execute()["name"]
            return os.path.join("/content/drive/Shareddrives", nombre, *reversed(partes))
        partes.append(f["name"])
        if not f.get("parents"):
            print(f'La carpeta "{partes[0]}" te la compartieron y no está en tu Mi unidad: en drive.google.com dale '
                  'clic derecho → Organizar → Añadir acceso directo → Mi unidad, y vuelve a darle Ejecutar todo '
                  '(con el link vacío; la encuentra por nombre).')
            raise SystemExit
        actual = f["parents"][0]

_h = [carpeta_del_link(LINK_DE_TU_CARPETA.strip())] if LINK_DE_TU_CARPETA.strip() else hallar_carpeta()
if _h and not os.path.isdir(_h[0]):
    print("No veo esa carpeta en tu Drive:", _h[0]); raise SystemExit
if not _h:
    print(f'No encontré la carpeta "{CARPETA}" en tu Drive. Si te la compartieron, en drive.google.com dale clic '
          'derecho → Organizar → Añadir acceso directo → Mi unidad, y vuelve a darle Ejecutar todo. '
          'O cambia CARPETA arriba por el nombre exacto.')
    raise SystemExit
if len(_h) > 1:
    print("Hay más de una carpeta con ese nombre; uso la primera:", *_h, sep="\n  ")
DEST = _h[0]
print("Van a:", DEST)
TMP = "/content/tmp_shiurim"
HILOS = 6

plan = json.loads(gzip.decompress(urllib.request.urlopen(PLAN_URL, timeout=120).read()).decode("utf-8"))
os.makedirs(DEST, exist_ok=True); os.makedirs(TMP, exist_ok=True)

arbol = {}
for x in plan:
    for i in range(1, min(len(x["r"]), 3) + 1):
        arbol[tuple(x["r"][:i])] = arbol.get(tuple(x["r"][:i]), 0) + 1
with open(os.path.join(DEST, "ÍNDICE.txt"), "w", encoding="utf-8") as f:
    f.write("Shiurim de Otzar HaTorah, por tema (el número es cuántos shiurim hay en la carpeta)\n\n")
    f.write("\n".join("    " * (len(k) - 1) + f"{k[-1]}  ({n})" for k, n in sorted(arbol.items())) + "\n")

def ruta(x):
    return os.path.join(DEST, *x["r"], x["n"])

def ya_esta(x):
    try:
        t = os.path.getsize(ruta(x))
    except OSError:
        return False
    return t > 0 and (not x["b"] or t == x["b"])

falta = [x for x in plan if not ya_esta(x)]
gb = lambda n: f"{n/1e9:.1f} GB"
total = sum(x["b"] or 0 for x in falta)
libre = shutil.disk_usage("/content/drive/MyDrive").free
print(f"{len(plan)} shiurim · ya están {len(plan)-len(falta)} · faltan {len(falta)} ({gb(total)}) · libre en tu Drive: {gb(libre)}")
if total > libre:
    print("OJO: no alcanza el espacio de tu Drive; bajo lo que quepa. Amplía en one.google.com y vuelve a correr.")

lock = threading.Lock()
hechos = {"n": 0, "b": 0, "err": []}
inicio = time.time()

def uno(i, x):
    tmp = os.path.join(TMP, f"{i}.part")
    ultimo = ""
    for intento in range(3):
        try:
            if shutil.disk_usage("/content/drive/MyDrive").free < (x["b"] or 0) + 1e9:
                raise RuntimeError("Drive lleno")
            req = urllib.request.Request(x["u"], headers={"User-Agent": "otzar-drive/1.0"})
            with urllib.request.urlopen(req, timeout=180) as r, open(tmp, "wb") as f:
                shutil.copyfileobj(r, f, 1 << 20)
            real = os.path.getsize(tmp)
            if real == 0 or (x["b"] and real < 0.9 * x["b"]):
                raise RuntimeError(f"incompleto ({real} de {x['b']})")
            os.makedirs(os.path.dirname(ruta(x)), exist_ok=True)
            shutil.copyfile(tmp, ruta(x))
            os.remove(tmp)
            with lock:
                hechos["n"] += 1; hechos["b"] += x["b"] or real
                if hechos["n"] % 50 == 0:
                    v = hechos["b"] / max(time.time() - inicio, 1)
                    print(f"{hechos['n']}/{len(falta)} · {gb(hechos['b'])} · faltan unas {(total-hechos['b'])/v/3600:.1f} h" if v else "")
            return
        except Exception as e:
            ultimo = e
            if os.path.exists(tmp):
                os.remove(tmp)
            if "Drive lleno" in str(e):
                break
            time.sleep(10 * (intento + 1))
    with lock:
        hechos["err"].append(f"{'/'.join(x['r'])}/{x['n']}  ←  {x['u']}  ({ultimo})")

with cf.ThreadPoolExecutor(HILOS) as ex:
    list(ex.map(lambda a: uno(*a), enumerate(falta)))

drive.flush_and_unmount()
print(f"\nListo: {hechos['n']} nuevos ({gb(hechos['b'])}).")
if hechos["err"]:
    print(f"{len(hechos['err'])} no se pudieron (vuelve a darle Ejecutar todo para reintentarlos):")
    print("\n".join(hechos["err"][:30]))
else:
    print(f"Todo está en {DEST.replace('/content/drive/MyDrive', 'Mi unidad')}.")